# 04 · Khoảng cách giữa dataset và đường thật (video tự quay trên xe máy)

**CPU là đủ** (chạy ONNX bằng onnxruntime). Kaggle: Settings → Accelerator → None (CPU). Colab: runtime CPU mặc định là được.

Input: code + models (notebook 03) + dataset riêng tư `motorbike-frames` gồm:
- `images/*.jpg` (tách bằng `python -m dashcam.frames`, đã làm mờ mặt)
- `labels/*.txt` (nhãn YOLO 52 lớp, gán bằng Label Studio/CVAT, xem docs/quay_video_xe_may.md)
- `frames.csv` (thời điểm, khu vực, độ nét của từng frame)

Trên Colab: copy thư mục `motorbike-frames` lên Drive vào `MyDrive/ai-portfolio/vn-dashcam-vision/`.

Câu hỏi: model train trên VNTS (đa phần ảnh từ ô tô / camera cố định) tụt bao nhiêu khi nhìn
từ xe máy, và tụt theo yếu tố nào (đêm, mưa, rung)?

In [ ]:
import glob, json, os, shutil, sys
import numpy as np, pandas as pd

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vn-dashcam-vision" if IN_COLAB else os.path.abspath("vn-dashcam-vision")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vn-dashcam-vision {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vn-dashcam-vision" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


os.makedirs("models", exist_ok=True)
if IN_KAGGLE:
    for f in glob.glob("/kaggle/input/**/models/*", recursive=True):
        shutil.copy(f, "models/")
else:  # Colab: models/ của notebook 01, 02 đã nằm trong WORK
    find("models/detector_1cls.onnx", "chạy notebook 01 trước")
    find("models/signnet.onnx", "chạy notebook 02 trước")
MB = os.path.dirname(find("frames.csv", f"Colab: copy thư mục motorbike-frames vào {WORK}"))
!pip install -q onnxruntime opencv-python-headless

In [ ]:
import cv2
from pathlib import Path
from dashcam.evaluate import load_gt
from dashcam.map_eval import map50
from dashcam.pipeline import SignPipeline

meta = pd.read_csv(f"{MB}/frames.csv")
meta["blur"] = pd.qcut(meta["sharpness"], 3, labels=["nhoè", "vừa", "nét"])

def run(mode):
    pipe_preds, gts = {}, {}
    for _, r in meta.iterrows():
        p = Path(MB) / "images" / r["file"]
        img = cv2.imread(str(p))
        res = SignPipeline.load("models", mode=mode).process(img)
        pipe_preds[p.stem] = [(b, float(s) * float(pr.max()), int(pr.argmax()))
                              for b, s, pr in zip(res.boxes, res.det_scores, res.probs) if pr is not None]
        gts[p.stem] = load_gt(p, Path(MB) / "labels", img.shape)
    return pipe_preds, gts

rows = []
for mode in ("one_stage", "two_stage"):
    preds, gts = run(mode)
    rows.append({"mode": mode, "nhóm": "tất cả", "mAP50": map50(preds, gts)[0]})
    for col in ("time", "area", "blur"):
        for val, sub in meta.groupby(col):
            keys = [Path(f).stem for f in sub["file"]]
            m, _ = map50({k: preds[k] for k in keys}, {k: gts[k] for k in keys})
            rows.append({"mode": mode, "nhóm": f"{col}={val}", "mAP50": round(m, 4), "n": len(keys)})
pd.DataFrame(rows)